# Arcadia run 5: bigger reading list, five voices, ~25M parameters

Trains `configs/run5.py` on the corpus built locally (`out/run5/corpus_table.txt`), uploaded as the private dataset **danieltgregory/arcadia-run5-data** (train/val splits, the `bpe8000_run5` tokenizer, and run 4's val set as the fixed benchmark). Also attach **danieltgregory/arcadia-run4-model** so run 4 can be scored on the same benchmark.

Settings: **GPU T4**, **Internet on**. Output: `/kaggle/working/run5/` (checkpoints, loss log, `eval.json`, samples).

## 1. Clone the repo and install what's missing

In [ ]:
import os, importlib.util, subprocess, sys
if not os.path.exists("/kaggle/working/Arcadia"):
    !git clone --depth 1 -b main https://github.com/DanielGregory/Arcadia /kaggle/working/Arcadia
%cd /kaggle/working/Arcadia
!git log --oneline -1
missing = [p for p in ("sentencepiece",) if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

## 2. Put the dataset where training expects it (data/run5/)

In [ ]:
import glob, shutil, torch
src = os.path.dirname(glob.glob("/kaggle/input/**/bpe8000_run5.model", recursive=True)[0])
os.makedirs("data/run5/tokenizer", exist_ok=True)
for f in glob.glob(f"{src}/*.txt"):
    if not os.path.basename(f).startswith("run4_"):
        shutil.copy(f, "data/run5/")
shutil.copy(f"{src}/bpe8000_run5.model", "data/run5/tokenizer/")
RUN4_VAL = f"{src}/run4_val.txt"
RUN4_CKPT = glob.glob("/kaggle/input/**/run4_best.pt", recursive=True)[0]
print(sorted(os.listdir("data/run5")), RUN4_VAL, RUN4_CKPT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 3. Train (up to 8000 steps; stops early after 2 evals without a new best)

In [ ]:
subprocess.run([sys.executable, "-m", "engine.train", "--config", "configs/run5.py", "--out_dir", "/kaggle/working/run5"], check=True)

## 4. Fair comparison

Run 4 and run 5 on **run 4's val set** with every tag removed (same plain text for both), then run 5 on its own val set with tags. Bits per character, overall and per tag.

In [ ]:
subprocess.run([sys.executable, "-m", "engine.evaluate", "--ckpt", RUN4_CKPT, "/kaggle/working/run5/best.pt",
                "--val", RUN4_VAL, "--out", "/kaggle/working/run5/eval_run4_benchmark.json"], check=True)
subprocess.run([sys.executable, "-m", "engine.evaluate", "--ckpt", "/kaggle/working/run5/best.pt",
                "--val", "data/run5/val.txt", "--keep_tags", "--out", "/kaggle/working/run5/eval_run5_val.json"], check=True)

## 5. Samples in every voice

In [ ]:
with open("/kaggle/working/run5/samples.txt", "w") as f:
    for tag in ["dream", "myth", "tale", "poem", "wisdom", "chronicle"]:
        out = subprocess.run([sys.executable, "-m", "engine.sample", "--ckpt", "/kaggle/working/run5/best.pt", "--dreams", "3",
                              "--tag", tag, "--tokens", "250", "--temperature", "0.9"], capture_output=True, text=True, check=True).stdout
        f.write(f"===== {tag} =====\n{out}\n")
        print(f"===== {tag} =====\n{out[:1500]}")